
# Task A XGBoost Classifier + Optuna

This notebook adds a **single-model `xgboost.XGBClassifier`** to the existing Task A workflow without changing preprocessing.

## What it reuses from the current Task A ensemble
- raw train / validation / test I/O from `credit_train.csv`, `credit_test.csv`, and `sample_submission.csv`
- the existing Task A preprocessing contract (`fit_task_a_preprocessor` / `transform_task_a`)
- the existing submission behavior that preserves `InterestRate` from `submissions/submission.csv` when available

## Assumptions about the current classifier interface
- Task A models accept raw pandas `DataFrame` inputs and integer `RiskTier` targets.
- Preprocessing is fit on training data only and then reused for validation / test, so leakage must stay out of the flow.
- Validation metrics should include accuracy, macro F1, confusion matrix, and a full classification report.
- The current task is **multiclass** (`RiskTier` in `[0, 1, 2, 3, 4]`), so `scale_pos_weight` is left disabled by default and is only meaningful if the task ever becomes binary.

## Dependencies
Install these in the notebook environment before running training or tuning:

```bash
pip install xgboost optuna joblib
```


In [16]:

from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        module_path = candidate / 'task_a' / 'scripts' / 'taskA_xgb_optuna.py'
        data_path = candidate / 'data' / 'creditsense-ai1215' / 'credit_train.csv'
        if module_path.exists() and data_path.exists():
            return candidate
    raise FileNotFoundError('Could not resolve the project root for Task A notebook I/O.')

REPO_ROOT = resolve_repo_root()
DATA_DIR = REPO_ROOT / 'data' / 'creditsense-ai1215'
ARTIFACT_DIR = REPO_ROOT / 'task_a' / 'artifacts'
REPORT_DIR = REPO_ROOT / 'task_a' / 'reports'
SUBMISSION_PATH = REPO_ROOT / 'submissions' / 'submission.csv'
MODULE_PATH = REPO_ROOT / 'task_a' / 'scripts' / 'taskA_xgb_optuna.py'

spec = importlib.util.spec_from_file_location('taskA_xgb_optuna', MODULE_PATH)
taska = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = taska
assert spec.loader is not None
spec.loader.exec_module(taska)

# Match the same repo-relative data and submission contract used by improved_taskA.ipynb.
taska.REPO_ROOT = REPO_ROOT
taska.DATA_DIR = DATA_DIR
taska.TRAIN_PATH = DATA_DIR / 'credit_train.csv'
taska.TEST_PATH = DATA_DIR / 'credit_test.csv'
taska.SAMPLE_SUBMISSION_PATH = DATA_DIR / 'sample_submission.csv'
taska.ARTIFACT_DIR = ARTIFACT_DIR
taska.REPORT_DIR = REPORT_DIR
taska.SUBMISSION_PATH = SUBMISSION_PATH

print('Loaded module:', MODULE_PATH)
print('Train CSV:', taska.TRAIN_PATH)
print('Test CSV :', taska.TEST_PATH)
print('Submission target:', taska.SUBMISSION_PATH)
print('Artifact dir:', taska.ARTIFACT_DIR)


Loaded module: /Users/dachi.tchotashvili/local docs/VS_main/ML final/ML-group-project/task_a/scripts/taskA_xgb_optuna.py
Train CSV: /Users/dachi.tchotashvili/local docs/VS_main/ML final/ML-group-project/data/creditsense-ai1215/credit_train.csv
Test CSV : /Users/dachi.tchotashvili/local docs/VS_main/ML final/ML-group-project/data/creditsense-ai1215/credit_test.csv
Submission target: /Users/dachi.tchotashvili/local docs/VS_main/ML final/ML-group-project/submissions/submission.csv
Artifact dir: /Users/dachi.tchotashvili/local docs/VS_main/ML final/ML-group-project/task_a/artifacts


In [ ]:

TRAIN_CONFIG = taska.TrainEvalConfig(
    artifact_tag='taskA_xgb_trainval',
    validation_size=0.2,
    random_state=42,
    model_threads=2,
    metric_name='accuracy',
    enable_early_stopping=True,
    early_stopping_rounds=40,
    internal_early_stopping_size=0.1,
    run_full_train_prediction=True,
    preserve_interest_rate=True,
    xgb_params={
        "n_estimators": 1035,
        "max_depth": 6,
        "learning_rate": 0.03992306657672681,
        "min_child_weight": 4.442311657343007,
        "subsample": 0.8524337374155861,
        "colsample_bytree": 0.8837262828873732,
        "gamma": 1.1909377972289308,
        "reg_alpha": 0.015936847132187043,
        "reg_lambda": 0.0318485426459654
    },
    use_scale_pos_weight=False,
)

print(json.dumps(taska.asdict(TRAIN_CONFIG), indent=2, default=taska._json_default))


{
  "artifact_tag": "taskA_xgb_trainval",
  "validation_size": 0.2,
  "random_state": 42,
  "model_threads": 2,
  "verbose": false,
  "metric_name": "accuracy",
  "enable_early_stopping": true,
  "early_stopping_rounds": 40,
  "internal_early_stopping_size": 0.1,
  "run_full_train_prediction": true,
  "preserve_interest_rate": true,
  "save_model_bundle": true,
  "xgb_params": {
    "n_estimators": 150,
    "max_depth": 4,
    "learning_rate": 0.05,
    "min_child_weight": 4.442311657343007,
    "subsample": 0.8524337374155861,
    "colsample_bytree": 0.8837262828873732,
    "gamma": 1.1909377972289308,
    "reg_alpha": 0.015936847132187043,
    "reg_lambda": 0.0318485426459654
  },
  "use_scale_pos_weight": false
}


In [35]:

RUN_VALIDATION = True

if RUN_VALIDATION:
    train_results = taska.run_train_validation_experiment(TRAIN_CONFIG)
    print(json.dumps(train_results, indent=2, default=taska._json_default))
else:
    print('Validation run skipped. Set RUN_VALIDATION=True to train and save artifacts.')


{
  "accuracy": 0.692,
  "macro_f1": 0.6892070379369408,
  "selected_score": 0.692,
  "elapsed_seconds": 2.01,
  "fit_metadata": {
    "early_stopping_used": true,
    "early_stopping_source": "external_validation",
    "best_iteration": 149,
    "best_n_estimators": 150,
    "feature_count": 115,
    "scale_pos_weight": null,
    "evals_result": {
      "validation_0": {
        "mlogloss": [
          1.578506632915565,
          1.5473128033535821,
          1.5185557858433043,
          1.491917227404458,
          1.467622827142477,
          1.444985452996833,
          1.4237435916257757,
          1.4042469006755522,
          1.3848298538923263,
          1.3665944549377476,
          1.3497722092526299,
          1.3330011409606253,
          1.316918782217162,
          1.3016691015703337,
          1.2880853536150285,
          1.2736760670649154,
          1.2609987806315932,
          1.2480766311245306,
          1.2361427409074137,
          1.224525892042688,
         

In [19]:

OPTUNA_CONFIG = taska.OptunaConfig(
    study_name='taskA_xgb_optuna',
    artifact_tag='taskA_xgb_optuna',
    n_trials=25,
    timeout=None,
    cv_folds=5,
    metric_name='accuracy',
    random_state=42,
    model_threads=2,
    enable_early_stopping=True,
    early_stopping_rounds=40,
    internal_early_stopping_size=0.1,
    include_scale_pos_weight=False,
    fixed_params={},
)

print(json.dumps(taska.asdict(OPTUNA_CONFIG), indent=2, default=taska._json_default))


{
  "study_name": "taskA_xgb_optuna",
  "artifact_tag": "taskA_xgb_optuna",
  "n_trials": 25,
  "timeout": null,
  "cv_folds": 5,
  "metric_name": "accuracy",
  "random_state": 42,
  "model_threads": 2,
  "verbose": false,
  "enable_early_stopping": true,
  "early_stopping_rounds": 40,
  "internal_early_stopping_size": 0.1,
  "include_scale_pos_weight": false,
  "fixed_params": {}
}


In [ ]:

RUN_TUNING = False

if RUN_TUNING:
    tuning_results = taska.run_optuna_study(OPTUNA_CONFIG)
    print(json.dumps(tuning_results, indent=2, default=taska._json_default))
else:
    print('Optuna run skipped. Set RUN_TUNING=True to start Bayesian optimization.')


[I 2026-04-26 15:55:37,319] A new study created in memory with name: taskA_xgb_optuna
[I 2026-04-26 15:57:10,635] Trial 0 finished with value: 0.775 and parameters: {'n_estimators': 574, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'min_child_weight': 7.585243326167403, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469887, 'reg_lambda': 0.44021796308748273}. Best is trial 0 with value: 0.775.
[I 2026-04-26 15:58:13,899] Trial 1 finished with value: 0.7752857142857142 and parameters: {'n_estimators': 908, 'max_depth': 3, 'learning_rate': 0.27081608642499677, 'min_child_weight': 10.156869048804639, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'gamma': 0.9170225492671691, 'reg_alpha': 5.472429642032189e-06, 'reg_lambda': 0.20316425758330103}. Best is trial 1 with value: 0.7752857142857142.
[I 2026-04-26 15:59:31,598] Trial 2 finished with value: 0.7874857142857142 and param

{
  "study_name": "taskA_xgb_optuna",
  "metric_name": "accuracy",
  "best_value": 0.7943714285714285,
  "best_params": {
    "n_estimators": 1035,
    "max_depth": 6,
    "learning_rate": 0.03992306657672681,
    "min_child_weight": 4.442311657343007,
    "subsample": 0.8524337374155861,
    "colsample_bytree": 0.8837262828873732,
    "gamma": 1.1909377972289308,
    "reg_alpha": 0.015936847132187043,
    "reg_lambda": 0.0318485426459654
  },
  "best_user_attrs": {
    "mean_accuracy": 0.7943714285714285,
    "mean_macro_f1": 0.7950481876151049,
    "mean_selected_score": 0.7943714285714285,
    "fold_rows": [
      {
        "fold": 1,
        "accuracy": 0.797,
        "macro_f1": 0.7979540503590178,
        "selected_score": 0.797,
        "best_n_estimators": 1032
      },
      {
        "fold": 2,
        "accuracy": 0.7952857142857143,
        "macro_f1": 0.7956700792250203,
        "selected_score": 0.7952857142857143,
        "best_n_estimators": 1035
      },
      {
       

In [ ]:

import math
import optuna
import pandas as pd
from optuna.distributions import FloatDistribution, IntDistribution
from optuna.trial import TrialState, create_trial
from plotly.subplots import make_subplots

optuna.logging.set_verbosity(optuna.logging.WARNING)

trials_csv = taska.build_artifact_paths(OPTUNA_CONFIG.artifact_tag)['optuna_trials_csv']
trials_df = pd.read_csv(trials_csv)
completed_df = trials_df.loc[trials_df['state'] == 'COMPLETE'].copy()

if completed_df.empty:
    raise ValueError(f'No completed Optuna trials found in {trials_csv}.')

param_distributions = {
    'n_estimators': IntDistribution(200, 1200),
    'max_depth': IntDistribution(3, 10),
    'learning_rate': FloatDistribution(1e-2, 3e-1, log=True),
    'min_child_weight': FloatDistribution(1.0, 12.0),
    'subsample': FloatDistribution(0.6, 1.0),
    'colsample_bytree': FloatDistribution(0.6, 1.0),
    'gamma': FloatDistribution(0.0, 5.0),
    'reg_alpha': FloatDistribution(1e-8, 10.0, log=True),
    'reg_lambda': FloatDistribution(1e-3, 25.0, log=True),
}

if OPTUNA_CONFIG.include_scale_pos_weight and 'params_scale_pos_weight' in completed_df.columns:
    param_distributions['scale_pos_weight'] = FloatDistribution(0.5, 10.0, log=True)

available_params = [
    name for name in param_distributions
    if f'params_{name}' in completed_df.columns and completed_df[f'params_{name}'].notna().any()
]

study = optuna.create_study(
    direction='maximize',
    study_name=f'{OPTUNA_CONFIG.study_name}_slice_rebuilt',
)

for _, row in completed_df.iterrows():
    params = {}
    distributions = {}
    for name in available_params:
        col = f'params_{name}'
        if pd.isna(row[col]):
            continue
        dist = param_distributions[name]
        value = int(row[col]) if isinstance(dist, IntDistribution) else float(row[col])
        params[name] = value
        distributions[name] = dist

    trial = create_trial(
        params=params,
        distributions=distributions,
        value=float(row['value']),
        state=TrialState.COMPLETE,
        user_attrs={
            'mean_accuracy': float(row['user_attrs_mean_accuracy']),
            'mean_macro_f1': float(row['user_attrs_mean_macro_f1']),
            'mean_selected_score': float(row['user_attrs_mean_selected_score']),
        },
    )
    study.add_trial(trial)

n_cols = 3
n_rows = math.ceil(len(available_params) / n_cols)
target_fn = lambda trial: trial.user_attrs.get('mean_accuracy', trial.value)

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=available_params,
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
)

for index, param_name in enumerate(available_params):
    row = index // n_cols + 1
    col = index % n_cols + 1
    slice_fig = optuna.visualization.plot_slice(
        study,
        params=[param_name],
        target=target_fn,
        target_name='Mean CV Accuracy',
    )
    for trace in slice_fig.data:
        fig.add_trace(trace, row=row, col=col)

    fig.update_xaxes(
        title_text=param_name,
        type=slice_fig.layout.xaxis.type,
        row=row,
        col=col,
    )
    fig.update_yaxes(title_text='Mean CV Accuracy', row=row, col=col)

fig.update_layout(
    height=max(100, 500 * n_rows),
    width=1500,
    showlegend=False,
)
fig.show()



## Example terminal commands

Train / validate and then generate a fresh submission using the full training set:

```bash
python task_a/scripts/taskA_xgb_optuna.py train           --artifact-tag taskA_xgb_trainval           --validation-size 0.2           --metric-name accuracy           --early-stopping-rounds 40           --xgb-param n_estimators=600           --xgb-param max_depth=6           --xgb-param learning_rate=0.05
```

Run Optuna tuning on the training data only:

```bash
python task_a/scripts/taskA_xgb_optuna.py tune           --study-name taskA_xgb_optuna           --artifact-tag taskA_xgb_optuna           --n-trials 40           --cv-folds 5           --metric-name accuracy
```

After tuning, reuse the best params JSON saved under `task_a/artifacts/` by loading it into `TRAIN_CONFIG.xgb_params`.

## Saved artifacts

A validation run writes:
- validation metrics JSON
- validation predictions CSV
- XGBoost model JSON
- preprocessor JSON
- joblib bundle metadata
- markdown report in `task_a/reports/`
- submission CSV in both `submissions/` and `task_a/artifacts/` when full-train prediction is enabled
